# Hypothesis Testing and Confidence Intervals

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 03.07 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/08_hypothesis_testing_confidence.ipynb)

---

## 🎯 Learning Objective

Learn t-tests, p-values, and confidence intervals as tools for deciding whether an observed effect is likely real or due to chance.

---

## 📐 Theory

Once you've fit a model or measured an effect, a natural question follows: is this real, or
could it plausibly be noise? Hypothesis testing formalizes that question.

### The hypothesis-testing framework

A **null hypothesis** $H_0$ states "no effect" (e.g. two models have equal accuracy); the
**alternative** $H_1$ states the effect you suspect exists. A **test statistic** $T$ is
computed from the data, and its distribution *under $H_0$* tells you how surprising the
observed value is. Two error types matter: a **Type I error** (false positive) rejects a true
$H_0$; a **Type II error** (false negative) fails to reject a false $H_0$. The **significance
level** $\alpha$ (conventionally 0.05) is the Type I error rate you're willing to accept.

### The t-test

The **one-sample t-test** checks whether a sample mean differs significantly from a hypothesized
value $\mu_0$, using the statistic

$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}},$$

where $\bar x$ is the sample mean, $s$ the sample standard deviation, $n$ the sample size — the
denominator is the **standard error**, the estimated standard deviation of $\bar x$ itself,
following from `03.03`'s $\text{Var}(\text{mean of }n\text{ iid}) = \sigma^2/n$. Under $H_0$,
$T$ follows a **Student's t-distribution** with $n-1$ degrees of freedom (heavier-tailed than
Gaussian, correcting for estimating $\sigma$ from the same finite sample). The **two-sample
t-test** runs the identical logic on the *difference* of two sample means, and is the standard
tool for **A/B testing**: is variant B's mean metric meaningfully different from variant A's?

### p-values

The **p-value** is $P(|T| \ge |t_{\text{observed}}| \mid H_0)$ — the probability, *if $H_0$ were
true*, of seeing a test statistic at least this extreme purely from sampling noise. A small
p-value means the observed data would be a very unlikely coincidence under $H_0$, evidence
against it. A p-value is **not** $P(H_0\mid\text{data})$ (that's a posterior, needing Bayes'
theorem from `03.05` and a prior) — conflating the two is one of the most common statistical
errors. Rejecting $H_0$ when $p<\alpha$ controls the Type I error rate at exactly $\alpha$, by
construction.

### Confidence intervals

A **95% confidence interval** for a parameter is an interval built by a procedure that, if
repeated over infinitely many fresh samples, contains the true parameter value 95% of the time.
For a mean with unknown $\sigma$, it's $\bar x \pm t^*_{n-1}\cdot s/\sqrt n$, where $t^*_{n-1}$
is the critical value cutting off $\alpha/2$ in each tail of the $t_{n-1}$ distribution. The
95% describes the *procedure's* long-run reliability, not "a 95% chance the true value is in
this particular interval" — a subtle but important distinction, since after the interval is
computed the true parameter either is or isn't in it.

### Multiple comparisons

Testing many hypotheses at once inflates the *overall* false-positive rate: with $m$ independent
tests each at level $\alpha$, the probability of at least one false positive is $1-(1-\alpha)^m$,
growing quickly with $m$. The **Bonferroni correction** counteracts this by testing each
individual hypothesis at level $\alpha/m$, keeping the family-wise error rate at $\alpha$ —
essential whenever a model is evaluated across many metrics, slices, or ablations at once.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The t-distribution under $H_0$, with the observed test statistic marked and its two-tailed
rejection region shaded — this is literally what a p-value measures: the shaded area's total
probability mass.

In [ ]:
# Two model variants' per-example accuracy scores (a small A/B test) -- is variant B really better?
rng = np.random.default_rng(0)
scores_A = rng.normal(0.82, 0.10, size=30)   # baseline model, accuracy-like scores per test batch
scores_B = rng.normal(0.87, 0.10, size=30)   # candidate model

t_stat, p_value = stats.ttest_ind(scores_B, scores_A)
df = len(scores_A) + len(scores_B) - 2

t_grid = np.linspace(-5, 5, 500)
t_pdf = stats.t(df).pdf(t_grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_grid, t_pdf, color="#4C72B0", lw=2, label=f"t-distribution, df={df} (H0: no difference)")
reject_region = np.abs(t_grid) >= np.abs(t_stat)
ax.fill_between(t_grid[reject_region], t_pdf[reject_region], color="#C44E52", alpha=0.5,
                 label=f"|t| >= |t_observed|  (total area = p-value = {p_value:.4f})")
ax.axvline(t_stat, color="black", ls="--", label=f"t_observed = {t_stat:.3f}")
ax.set_xlabel("t"); ax.set_ylabel("density under H0"); ax.legend(fontsize=9)
ax.set_title("Two-sample t-test: how extreme is the observed difference, under 'no real effect'?")
plt.tight_layout()
plt.show()

print(f"Mean(A)={scores_A.mean():.4f}, Mean(B)={scores_B.mean():.4f}, observed difference={scores_B.mean()-scores_A.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
print(f"At alpha=0.05, reject H0 (no difference)? {p_value < 0.05}")

## 🐍 Implementation from Scratch

We compute the two-sample t-statistic and p-value directly from their definitions (no
`scipy.stats.ttest_ind` shortcut), build a confidence interval by hand, and verify both against
`scipy.stats`.

In [ ]:
# Two-sample (Welch's) t-test from scratch, straight from the Theory section's formula
def two_sample_ttest(x, y):
    n_x, n_y = len(x), len(y)
    mean_x, mean_y = x.mean(), y.mean()
    var_x, var_y = x.var(ddof=1), y.var(ddof=1)          # ddof=1: unbiased sample variance
    se_diff = np.sqrt(var_x / n_x + var_y / n_y)          # standard error of the DIFFERENCE of means
    t_stat = (mean_y - mean_x) / se_diff
    # Welch-Satterthwaite degrees of freedom -- exact when variances differ between groups
    df = (var_x / n_x + var_y / n_y) ** 2 / (
        (var_x / n_x) ** 2 / (n_x - 1) + (var_y / n_y) ** 2 / (n_y - 1)
    )
    p_value = 2 * (1 - stats.t(df).cdf(np.abs(t_stat)))   # two-tailed: both directions count as "extreme"
    return t_stat, p_value, df

rng = np.random.default_rng(0)
scores_A = rng.normal(0.82, 0.10, size=30)
scores_B = rng.normal(0.87, 0.10, size=30)

t_ours, p_ours, df_ours = two_sample_ttest(scores_A, scores_B)
t_scipy, p_scipy = stats.ttest_ind(scores_B, scores_A, equal_var=False)  # Welch's t-test, matching ours

print(f"Our t-statistic: {t_ours:.6f}   scipy: {t_scipy:.6f}   diff: {abs(t_ours - t_scipy):.2e}")
print(f"Our p-value:     {p_ours:.6f}   scipy: {p_scipy:.6f}   diff: {abs(p_ours - p_scipy):.2e}")
print(f"Our df:          {df_ours:.4f} (Welch-Satterthwaite)")

# A 95% confidence interval for the mean of scores_A, built from the t-distribution by hand
def confidence_interval_mean(data, confidence=0.95):
    n = len(data)
    mean, sem = data.mean(), data.std(ddof=1) / np.sqrt(n)
    t_critical = stats.t(n - 1).ppf(1 - (1 - confidence) / 2)  # two-tailed critical value
    return mean - t_critical * sem, mean + t_critical * sem

ci_low, ci_high = confidence_interval_mean(scores_A)
ci_scipy = stats.t.interval(0.95, df=len(scores_A) - 1, loc=scores_A.mean(),
                             scale=stats.sem(scores_A))
print(f"\nOur 95% CI for mean(A):    ({ci_low:.4f}, {ci_high:.4f})")
print(f"scipy.stats.t.interval:   ({ci_scipy[0]:.4f}, {ci_scipy[1]:.4f})")

# Empirically verify the CONFIDENCE INTERVAL'S actual coverage guarantee: repeat "sample n=30,
# build a 95% CI" thousands of times from a KNOWN true mean, and check ~95% of intervals contain it
true_mean_check, true_std_check = 0.82, 0.10
n_trials = 5000
coverage_count = 0
for _ in range(n_trials):
    sample = rng.normal(true_mean_check, true_std_check, size=30)
    lo, hi = confidence_interval_mean(sample)
    coverage_count += (lo <= true_mean_check <= hi)
print(f"\nEmpirical coverage over {n_trials} fresh samples: {coverage_count/n_trials:.4f}  (target: 0.95)")

## 🤖 Why This Matters for AI

Deciding whether a fine-tuning trick, architecture change, or hyperparameter *actually* improved
a model, versus just landing luckier on this particular validation split, is exactly a
hypothesis test. **Ablation studies** run this same t-test/confidence-interval logic across many
configurations at once, which is precisely where multiple-comparisons correction becomes
unavoidable — testing 20 ablations at $\alpha=0.05$ each gives roughly a 64% chance of at least
one false "improvement" by pure chance if none of them actually help.

In practice, experiment tracking platforms like **Weights & Biases** and **MLflow** automate exactly this analysis: they log metrics across runs, compute confidence intervals, and flag when an apparent improvement is not statistically significant. A/B testing platforms in recommendation systems (at Netflix, Spotify) run sequential hypothesis tests on user engagement metrics — the same t-test logic, applied at scale.

---

In [ ]:
# A realistic ablation study: 6 model variants compared against a baseline. Some genuinely
# help, some don't -- multiple comparisons correction changes which "wins" survive scrutiny.
rng = np.random.default_rng(0)
n_eval_examples = 40
baseline_scores = rng.normal(0.80, 0.08, size=n_eval_examples)

# True effects baked into the simulation: variants 0,1 give a real (small) improvement; the
# rest are NO-OP variants (same true mean as baseline) -- we don't get to see this in practice
true_deltas = [0.05, 0.04, 0.0, 0.0, 0.0, 0.0]
variant_scores = {
    f"variant_{i}": rng.normal(0.80 + delta, 0.08, size=n_eval_examples)
    for i, delta in enumerate(true_deltas)
}

raw_pvalues, deltas_observed = [], []
print(f"{'variant':>10} | {'mean diff':>10} | {'p-value':>9}")
for name, scores in variant_scores.items():
    t_stat, p_val = stats.ttest_ind(scores, baseline_scores, equal_var=False)
    raw_pvalues.append(p_val)
    deltas_observed.append(scores.mean() - baseline_scores.mean())
    print(f"{name:>10} | {deltas_observed[-1]:10.4f} | {p_val:9.4f}")

alpha = 0.05
uncorrected_wins = [p < alpha for p in raw_pvalues]
_, corrected_pvalues, _, _ = multipletests(raw_pvalues, alpha=alpha, method="bonferroni")
corrected_wins = corrected_pvalues < alpha

print(f"\nUncorrected 'significant' variants (p < {alpha}):  {[n for n, w in zip(variant_scores, uncorrected_wins) if w]}")
print(f"Bonferroni-corrected 'significant' variants:        {[n for n, w in zip(variant_scores, corrected_wins) if w]}")
print(f"\n(true improving variants were variant_0, variant_1 -- correction is designed to filter")
print(f" out false 'wins' from the no-op variants, at the cost of also needing a stronger")
print(f" signal to call a real-but-small effect significant)")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For the small sample $x=(5.2, 4.8, 5.5, 5.1, 4.9, 5.3)$, test $H_0:\mu=5.0$ by hand: compute
   the sample mean $\bar x$, sample standard deviation $s$ (with $n-1$ in the denominator), the
   standard error $s/\sqrt n$, the t-statistic, and the degrees of freedom $n-1$. As a sanity
   check, confirm your standard error is strictly positive (a negative or zero SE would signal
   an arithmetic error), then verify your t-statistic against `scipy.stats.ttest_1samp(x, 5.0)`.

<details>
<summary>💡 Solution</summary>

$\bar x = (5.2+4.8+5.5+5.1+4.9+5.3)/6 = 5.1\overline{3}\approx 5.1333$. Sample variance
$s^2=\frac{1}{n-1}\sum(x_i-\bar x)^2\approx 0.06667$, so $s\approx 0.2582$ — positive, as
required. Standard error $s/\sqrt{6}\approx 0.1054$. $t = (\bar x-\mu_0)/\text{SE} =
(5.1333-5.0)/0.1054 \approx 1.265$, with $df=n-1=5$. `scipy.stats.ttest_1samp(x, 5.0)` returns
`t≈1.265, p≈0.262`, matching the by-hand t-statistic exactly.

</details>

### 💭 UNDERSTAND
2. A colleague computes a p-value of $0.03$ for an A/B test and states: "there's a 97% chance
   variant B is really better than variant A." Explain precisely what is wrong with this
   statement, in terms of what a p-value actually conditions on.

<details>
<summary>💡 Solution</summary>

A p-value is $P(\text{data this extreme or more} \mid H_0\text{ is true})$ — a statement about
how surprising the *data* would be *if there were no real effect*. "97% chance B is really
better" would instead be a statement about $P(H_1\mid\text{data})$, the *posterior* probability
of the hypothesis given the data — a fundamentally different conditional that requires Bayes'
theorem (`03.05`) and a prior over how likely a real effect was *before* seeing any data. The
p-value says nothing at all about $P(H_0\mid\text{data})$ or $P(H_1\mid\text{data})$ directly;
conflating "$P(\text{data}\mid H_0)$ is small" with "$P(H_0\mid\text{data})$ is small" is exactly
the error the Theory section flags as one of the most common statistical mistakes.

</details>

### ✏️ DERIVE
3. Show that, for a two-sided test at level $\alpha$, "$H_0:\mu=\mu_0$ is rejected" and "$\mu_0$
   falls outside the $(1-\alpha)$ confidence interval for $\mu$" are the *same* statement — i.e.
   derive the equivalence between the t-test's rejection rule and the confidence interval's
   coverage rule, starting from their two definitions.

<details>
<summary>💡 Solution outline</summary>

The test rejects $H_0$ when $|t| = \left|\frac{\bar x - \mu_0}{s/\sqrt n}\right| > t^*_{n-1}$,
where $t^*_{n-1}$ is the critical value cutting off $\alpha/2$ in each tail. Multiply both sides
by $s/\sqrt n$ (positive, so the inequality direction is preserved):
$$|\bar x - \mu_0| > t^*_{n-1}\cdot \frac{s}{\sqrt n}.$$
This says $\mu_0$ lies outside the interval $\left(\bar x - t^*_{n-1}\frac{s}{\sqrt n},\ \bar x +
t^*_{n-1}\frac{s}{\sqrt n}\right)$ — but that interval is *exactly* the $(1-\alpha)$ confidence
interval for $\mu$ defined in the Theory section. So "$|t|$ exceeds the critical value" (reject
$H_0$) and "$\mu_0$ is outside the CI" are algebraically the identical condition. Numerically:
for a sample with $\bar x\approx5.13$, testing several candidate $\mu_0$ values against a fixed
95% CI shows the t-test's reject/fail-to-reject decision matches exactly whether that $\mu_0$
falls outside/inside the CI, for every value tried.

</details>

### 🐍 IMPLEMENT
4. Using `scores_A` from the notebook's own cells, construct a 95% confidence interval for its
   true mean by hand (using `stats.t.ppf` for the critical value, following
   `confidence_interval_mean`'s own pattern), and verify it matches
   `stats.t.interval(0.95, df=len(scores_A)-1, loc=scores_A.mean(), scale=stats.sem(scores_A))`.

<details>
<summary>✅ How to know you're right</summary>

Your by-hand interval's lower and upper bounds should match `stats.t.interval`'s output to
floating-point precision (both are computing $\bar x \pm t^*_{n-1}\cdot s/\sqrt n$ from the
identical data, just via different code paths) — with this notebook's own seeded `scores_A`,
the interval comes out to approximately $(0.777, 0.839)$. Any mismatch beyond about `1e-10`
indicates a slip in computing the critical value or the standard error, not a modeling
difference.

</details>

### 🤖 APPLY
5. Simulate a scenario where $H_0$ is actually **true** (draw both groups from the *identical*
   distribution, e.g. both `N(0.82, 0.10)` with $n=30$ each). Run the two-sample t-test on 1000
   independently-drawn fresh sample pairs and record how often $p<0.05$.

<details>
<summary>✅ What you should observe</summary>

The empirical false-positive rate across 1000 trials should land close to $5\%$ (a
representative run gives about $5.5\%$, well within sampling noise of the target for 1000
trials) — confirming the significance level $\alpha$ is not an arbitrary label but an exact
operational guarantee: when $H_0$ is genuinely true, the test rejects it at (very close to)
exactly the rate $\alpha$ was chosen to be, by the test's own construction.

</details>

### 🚀 CHALLENGE
6. Simulate 20 independent "ablations" against a shared baseline, where NONE actually change
   anything (every variant and the baseline drawn from the identical distribution
   `N(0.80, 0.08)`, $n=40$ each). Run a t-test for each variant against the *same* baseline
   sample, and across many repeated trials of this whole 20-test experiment, estimate (a) the
   probability that at least one of the 20 uncorrected tests falsely rejects $H_0$
   (family-wise error rate), and (b) the same probability after applying Bonferroni correction
   (`multipletests(pvals, alpha=0.05, method='bonferroni')`). Compare your empirical uncorrected
   FWER to the "independent tests" formula $1-(1-\alpha)^m$ from `03.01`/`03.08`'s Theory
   section, and explain any discrepancy you find — in particular, why comparing every one of the
   20 variants against the exact *same* baseline sample breaks a key assumption behind that
   formula.

<details>
<summary>🔍 What a strong answer covers</summary>

The Bonferroni-corrected family-wise error rate should land close to the target $5\%$ (a
representative run gives about $3$–$4\%$, comfortably controlled at or below $\alpha$). The
*uncorrected* empirical FWER, however, comes out noticeably **below** the formula's prediction
of $1-(0.95)^{20}\approx 64\%$ — representative runs give roughly $40$–$44\%$ instead. A strong
answer identifies why: the formula $1-(1-\alpha)^m$ assumes the $m$ tests are *independent*, but
here every one of the 20 tests is compared against the *same* shared baseline sample, so the 20
test statistics are correlated with each other (a baseline sample that happens to be unusually
high or low shifts every comparison's t-statistic in a correlated way) — positively correlated
test statistics make "at least one exceeds threshold" *less* likely than the independent-case
formula predicts, since the tests partly rise and fall together rather than offering 20 fully
independent chances to false-positive. A strong answer verifies this by checking the pairwise
correlation between two of the 20 variants' p-values across repeated trials (noticeably positive,
not zero) and connects the finding to real ablation studies: comparing many variants against one
shared baseline/validation set, as is standard practice, is exactly this correlated-tests
scenario, so naive application of the independent-tests multiple-comparisons formula can be
overly pessimistic — though Bonferroni correction itself remains valid (if conservative) even
under this correlation, since it does not rely on the independence assumption to control the
family-wise rate.

</details>

---

## 📚 Further Reading
- Wasserman, *All of Statistics*, Ch. 10 (Hypothesis Testing and p-values)
- Kohavi, Tang & Xu, *Trustworthy Online Controlled Experiments* (A/B testing at scale)
- Dror et al., ["The Hitchhiker's Guide to Testing Statistical Significance in Natural Language Processing"](https://aclanthology.org/P18-1128/)

---

*Next notebook: [Sampling and Monte Carlo Methods](09_sampling_and_monte_carlo.ipynb)*